## C-LSTM: Cross-National Bill Passage Prediction

Two separate C-LSTM models — one trained on US bills, one on Canadian bills — are evaluated both in-distribution and on each other's test sets.  
Input: `text_clean_nc` (country-neutral preprocessed text). Target: `label` (1 = passed).

In [3]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

ROOT = Path("..").resolve()
NORM = ROOT / "content" / "drive" / "MyDrive" / "UBC" / "2025W" / "Term2" / "CPSC440"
# NORM = ROOT / "data" / "normalized"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Requirements: tensorflow>=2.12, scikit-learn, pandas, numpy

import numpy as np
import pandas as pd

from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
)


### Load & Split Data

In [5]:
TEXT_COL    = "text_clean_nc"   # country-neutral text (no 'canada', 'congress', etc.)
LABEL_COL   = "passed"
COUNTRY_COL = "source"          # values: 'us' / 'canada'
YEAR_COL    = "year"

# Temporal cutoffs — train on older bills, test on the most recent years
# US:  ≤2022 train (~79%)  |  2023–2024 test (~21%)
# CA:  ≤2020 train (~76%)  |  2021–2026 test (~24%)
US_CUTOFF = 2022
CA_CUTOFF = 2020

df = pd.read_csv(NORM / "bills_clean.csv")
df = df[[TEXT_COL, LABEL_COL, COUNTRY_COL, YEAR_COL]].dropna().copy()
df[TEXT_COL]  = df[TEXT_COL].astype(str)
df[LABEL_COL] = df[LABEL_COL].astype(int)
df[YEAR_COL]  = df[YEAR_COL].astype(float).astype(int)

us = df[df[COUNTRY_COL] == "us"]
ca = df[df[COUNTRY_COL] == "canada"]

us_train = us[us[YEAR_COL] <= US_CUTOFF]
us_test  = us[us[YEAR_COL] >  US_CUTOFF]

ca_train = ca[ca[YEAR_COL] <= CA_CUTOFF]
ca_test  = ca[ca[YEAR_COL] >  CA_CUTOFF]

print(f"US  — train: {len(us_train):,} (≤{US_CUTOFF}), test: {len(us_test):,} ({US_CUTOFF+1}–present)")
print(f"      train pass rate: {us_train[LABEL_COL].mean():.3f}  |  test pass rate: {us_test[LABEL_COL].mean():.3f}")
print(f"\nCA  — train: {len(ca_train):,} (≤{CA_CUTOFF}), test: {len(ca_test):,} ({CA_CUTOFF+1}–present)")
print(f"      train pass rate: {ca_train[LABEL_COL].mean():.3f}  |  test pass rate: {ca_test[LABEL_COL].mean():.3f}")

/tmp/ipykernel_6476/2025635725.py:12: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(NORM / "bills_clean.csv")


US  — train: 59,329 (≤2022), test: 16,186 (2023–present)
      train pass rate: 0.026  |  test pass rate: 0.009

CA  — train: 4,663 (≤2020), test: 663 (2021–present)
      train pass rate: 0.166  |  test pass rate: 0.181


### Tokenization & Sequence Padding

In [7]:
MAX_FEATURES = 10_000
MAX_LEN      = 512

# Separate tokenizers per country — fitting on training data only prevents
# test-set vocabulary from leaking into the index mapping
tok_us = Tokenizer(num_words=MAX_FEATURES, oov_token="<OOV>")
tok_us.fit_on_texts(us_train[TEXT_COL])

tok_ca = Tokenizer(num_words=MAX_FEATURES, oov_token="<OOV>")
tok_ca.fit_on_texts(ca_train[TEXT_COL])


def encode(tokenizer: Tokenizer, texts, maxlen: int = MAX_LEN) -> np.ndarray:
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=maxlen, padding="post", truncating="post")


# In-distribution encodings
X_us_train = encode(tok_us, us_train[TEXT_COL])
X_us_test  = encode(tok_us, us_test[TEXT_COL])

X_ca_train = encode(tok_ca, ca_train[TEXT_COL])
X_ca_test  = encode(tok_ca, ca_test[TEXT_COL])

# Cross-domain: re-tokenize with the SOURCE model's tokenizer so the embedding
# index space matches what each model was trained on
X_ca_test_via_us = encode(tok_us, ca_test[TEXT_COL])   # US model → CA test
X_us_test_via_ca = encode(tok_ca, us_test[TEXT_COL])   # CA model → US test

y_us_train = us_train[LABEL_COL].values
y_us_test  = us_test[LABEL_COL].values
y_ca_train = ca_train[LABEL_COL].values
y_ca_test  = ca_test[LABEL_COL].values

print(f"X_us_train: {X_us_train.shape}")
print(f"X_ca_train: {X_ca_train.shape}")

X_us_train: (59329, 512)
X_ca_train: (4663, 512)


### C-LSTM Architecture

In [8]:
def build_clstm(max_features: int = MAX_FEATURES, embed_dim: int = 128, max_len: int = MAX_LEN) -> keras.Model:
    """
    C-LSTM: two Conv1D blocks compress the token sequence into a shorter
    feature map; an LSTM then models temporal dependencies over that map.

    Design notes:
    - Two MaxPooling1D(4) layers reduce sequence length from 10 000 → 625,
      making the LSTM tractable on long legislative texts.
    - LeakyReLU avoids dead-neuron saturation common with ReLU in sparse text.
    - BatchNorm before the LSTM stabilizes the distribution seen by recurrent
      gates, which are sensitive to input scale.
    - L1+L2 on LSTM kernel regularizes recurrent weights against overfitting
      on the relatively small corpus.
    """
    inputs = keras.Input(shape=(max_len,))

    x = layers.Embedding(max_features, embed_dim, input_length=max_len)(inputs)

    x = layers.Conv1D(64, kernel_size=3, padding="same")(x)
    x = layers.LeakyReLU()(x)
    x = layers.MaxPooling1D(4)(x)

    x = layers.Conv1D(32, kernel_size=3, padding="same")(x)
    x = layers.LeakyReLU()(x)
    x = layers.MaxPooling1D(4)(x)

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)

    x = layers.LSTM(32, kernel_regularizer=regularizers.l1_l2(l1=1e-5, l2=1e-4))(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model


build_clstm().summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 512, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 512, 64)        │        24,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 512, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 128, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 128, 32)        │         6,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 128, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 32, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 32)             │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,319,297 (5.03 MB)

 Trainable params: 1,319,233 (5.03 MB)

 Non-trainable params: 64 (256.00 B)

### Training

In [9]:
EPOCHS     = 30
BATCH_SIZE = 32

# patience=5 stops training when val_loss stops improving for 5 consecutive epochs
early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)


def get_class_weights(labels: np.ndarray) -> dict:
    # 'balanced' scales weights inversely proportional to class frequency,
    # compensating for the heavy class imbalance in legislative pass rates
    classes = np.unique(labels)
    weights = compute_class_weight("balanced", classes=classes, y=labels)
    return dict(zip(classes.tolist(), weights.tolist()))


# --- Train US model ---
print("=" * 50)
print("Training model_US")
print("=" * 50)
model_US = build_clstm()
cw_us = get_class_weights(y_us_train)
print(f"Class weights: {cw_us}")

history_US = model_US.fit(
    X_us_train, y_us_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    class_weight=cw_us,
    callbacks=[early_stop],
    verbose=1,
)

# --- Train CA model ---
print("\n" + "=" * 50)
print("Training model_CA")
print("=" * 50)
model_CA = build_clstm()
cw_ca = get_class_weights(y_ca_train)
print(f"Class weights: {cw_ca}")

history_CA = model_CA.fit(
    X_ca_train, y_ca_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    class_weight=cw_ca,
    callbacks=[early_stop],
    verbose=1,
)

Training model_US
Class weights: {0: 0.5136087400661392, 1: 18.87054707379135}
Epoch 1/30
1669/1669 ━━━━━━━━━━━━━━━━━━━━ 27s 12ms/step - accuracy: 0.5210 - loss: 0.6932 - val_accuracy: 0.4362 - val_loss: 0.8306
Epoch 2/30
1669/1669 ━━━━━━━━━━━━━━━━━━━━ 35s 11ms/step - accuracy: 0.7779 - loss: 0.5417 - val_accuracy: 0.8158 - val_loss: 0.4677
Epoch 3/30
1669/1669 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - accuracy: 0.8411 - loss: 0.4240 - val_accuracy: 0.8429 - val_loss: 0.3837
Epoch 4/30
1669/1669 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - accuracy: 0.8734 - loss: 0.3465 - val_accuracy: 0.8909 - val_loss: 0.3076
Epoch 5/30
1669/1669 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - accuracy: 0.9009 - loss: 0.2837 - val_accuracy: 0.9065 - val_loss: 0.2598
Epoch 6/30
1669/1669 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step - accuracy: 0.9103 - loss: 0.2433 - val_accuracy: 0.8712 - val_loss: 0.3506
Epoch 7/30
1669/1669 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - accuracy: 0.9191 - loss: 0.2191 - val_accuracy: 0.9327 - val_loss: 0.21

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


132/132 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.7521 - loss: 0.6141 - val_accuracy: 0.8137 - val_loss: 0.6434
Epoch 2/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8241 - loss: 0.5533 - val_accuracy: 0.8158 - val_loss: 0.5833
Epoch 3/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8701 - loss: 0.4800 - val_accuracy: 0.8009 - val_loss: 0.5012
Epoch 4/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8973 - loss: 0.3831 - val_accuracy: 0.6767 - val_loss: 0.7519
Epoch 5/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.8889 - loss: 0.3844 - val_accuracy: 0.6981 - val_loss: 0.6908


### Evaluation

In [10]:
def evaluate(model: keras.Model, X: np.ndarray, y_true: np.ndarray, label: str) -> dict:
    y_prob = model.predict(X, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        "Condition": label,
        "PR-AUC":    average_precision_score(y_true, y_prob),
        "ROC-AUC":   roc_auc_score(y_true, y_prob),
        "F1":        f1_score(y_true, y_pred, zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
    }


results = [
    evaluate(model_US, X_us_test,         y_us_test,  "US → US (in-dist) "),
    evaluate(model_CA, X_ca_test,         y_ca_test,  "CA → CA (in-dist) "),
    evaluate(model_US, X_ca_test_via_us,  y_ca_test,  "US → CA (transfer)"),
    evaluate(model_CA, X_us_test_via_ca,  y_us_test,  "CA → US (transfer)"),
]

results_df = pd.DataFrame(results).set_index("Condition")

# Pretty-print the summary table
header = f"{'Condition':<25}| {'PR-AUC':>7} | {'ROC-AUC':>7} | {'F1':>6} | {'Precision':>9} | {'Recall':>6}"
sep    = "-" * len(header)
print(sep)
print(header)
print(sep)
for cond, row in results_df.iterrows():
    print(f"{cond:<25}| {row['PR-AUC']:>7.3f} | {row['ROC-AUC']:>7.3f} | {row['F1']:>6.3f} | {row['Precision']:>9.3f} | {row['Recall']:>6.3f}")
print(sep)

--------------------------------------------------------------------------
Condition                |  PR-AUC | ROC-AUC |     F1 | Precision | Recall
--------------------------------------------------------------------------
US → US (in-dist)        |   0.082 |   0.795 |  0.099 |     0.055 |  0.461
CA → CA (in-dist)        |   0.442 |   0.725 |  0.422 |     0.312 |  0.650
US → CA (transfer)       |   0.163 |   0.416 |  0.099 |     0.190 |  0.067
CA → US (transfer)       |   0.007 |   0.380 |  0.012 |     0.006 |  0.237
--------------------------------------------------------------------------


### Save Models

In [13]:
model_US.save(NORM / "model_US.keras")
model_CA.save(NORM / "model_CA.keras")
print("Saved model_US.keras and model_CA.keras to", NORM)

Saved model_US.keras and model_CA.keras to /content/drive/MyDrive/UBC/2025W/Term2/CPSC440
